# FRED Series Bronze Ingestion

Heavy notebook entrypoint for ingesting FRED observations and series metadata into Bronze.

This notebook writes to:

- `brz_macro.raw_fred_series`
- `brz_macro.raw_fred_series_metadata`

Execution modes:

- `backfill`: explicit inclusive observation date range
- `incremental`: observation-window re-pull for phase-1 correctness

This notebook preserves FRED real-time revision windows via `(realtime_start, realtime_end)` and stores both `value_raw` and parsed numeric `value`.

Run `00_platform_setup_catalog_schema.ipynb` first.

In [ ]:
import hashlib
import json
import time
import uuid
from datetime import datetime, timedelta, timezone

import requests
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, DoubleType, StringType, StructField, StructType, TimestampType
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")

FRED_API_BASE = "https://api.stlouisfed.org/fred"
FRED_COMPLETE_REALTIME_START = "1776-07-04"
FRED_COMPLETE_REALTIME_END = "9999-12-31"
FRED_DEFAULT_LIMIT = 100000


def parse_iso_date(field_name: str, raw_value: str):
    try:
        return datetime.strptime(raw_value, "%Y-%m-%d").date()
    except ValueError as exc:
        raise ValueError(f"{field_name} must be in YYYY-MM-DD format") from exc


def parse_series_ids(raw_value: str) -> list[str]:
    series_ids = [item.strip().upper() for item in raw_value.split(",") if item.strip()]
    series_ids = list(dict.fromkeys(series_ids))
    if not series_ids:
        raise ValueError("series_ids cannot be empty")
    return series_ids


def resolve_date_window(mode: str, start_date_raw: str, end_date_raw: str, lookback_days_raw: str):
    latest_complete_date = datetime.now(timezone.utc).date() - timedelta(days=1)
    normalized_mode = mode.strip().lower()

    if normalized_mode not in {"backfill", "incremental"}:
        raise ValueError("mode must be either 'backfill' or 'incremental'")

    if normalized_mode == "backfill":
        if not start_date_raw or not end_date_raw:
            raise ValueError("backfill mode requires both start_date and end_date")
        start_date = parse_iso_date("start_date", start_date_raw)
        end_date = parse_iso_date("end_date", end_date_raw)
    else:
        try:
            lookback_days = int(lookback_days_raw or "0")
        except ValueError as exc:
            raise ValueError("lookback_days must be an integer in incremental mode") from exc

        if lookback_days < 1:
            raise ValueError("lookback_days must be >= 1 in incremental mode")

        end_date = parse_iso_date("end_date", end_date_raw) if end_date_raw else latest_complete_date
        start_date = end_date - timedelta(days=lookback_days - 1)

    if start_date > end_date:
        raise ValueError("start_date cannot be after end_date")

    if end_date > latest_complete_date:
        raise ValueError(
            f"end_date must be <= latest completed UTC day: {latest_complete_date.isoformat()}"
        )

    return start_date, end_date


def parse_optional_date(raw_value: str):
    if not raw_value:
        return None
    return datetime.strptime(raw_value, "%Y-%m-%d").date()


def request_json(session, endpoint: str, params: dict, max_retries: int = 5):
    url = f"{FRED_API_BASE}/{endpoint}"
    headers = {"Accept": "application/json", "User-Agent": "market-macro-lakehouse/phase1"}

    for attempt in range(max_retries):
        try:
            response = session.get(url, params=params, headers=headers, timeout=30)
            if response.status_code == 200:
                return response.json()

            if response.status_code == 429 or 500 <= response.status_code < 600:
                time.sleep((2 ** attempt) + 0.5)
                continue

            raise RuntimeError(f"FRED API error {response.status_code} for {endpoint}: {response.text[:500]}")
        except requests.RequestException as exc:
            if attempt == max_retries - 1:
                raise RuntimeError(f"FRED request failed for endpoint {endpoint}") from exc
            time.sleep((2 ** attempt) + 0.5)

    raise RuntimeError(f"Exhausted retries for FRED endpoint {endpoint}")


def fetch_series_metadata(session, api_key: str, series_id: str, run_id: str, ingested_at: datetime):
    payload = request_json(
        session,
        endpoint="series",
        params={
            "series_id": series_id,
            "api_key": api_key,
            "file_type": "json",
        },
    )

    series_items = payload.get("seriess", [])
    if not series_items:
        raise RuntimeError(f"No FRED series metadata returned for series_id={series_id}")

    series_item = series_items[0]
    payload_hash = hashlib.sha256(
        json.dumps(series_item, sort_keys=True, separators=(",", ":")).encode("utf-8")
    ).hexdigest()

    return {
        "series_id": series_id,
        "title": series_item.get("title"),
        "frequency": series_item.get("frequency"),
        "frequency_short": series_item.get("frequency_short"),
        "units": series_item.get("units"),
        "units_short": series_item.get("units_short"),
        "seasonal_adjustment": series_item.get("seasonal_adjustment"),
        "seasonal_adjustment_short": series_item.get("seasonal_adjustment_short"),
        "observation_start": parse_optional_date(series_item.get("observation_start")),
        "observation_end": parse_optional_date(series_item.get("observation_end")),
        "last_updated": series_item.get("last_updated"),
        "notes": series_item.get("notes"),
        "ingested_at": ingested_at,
        "run_id": run_id,
        "payload_hash": payload_hash,
    }


def parse_value(value_raw: str):
    if value_raw is None:
        return None
    normalized = value_raw.strip()
    if normalized in {"", "."}:
        return None
    return float(normalized)


def fetch_series_observations(session, api_key: str, series_id: str, start_date, end_date, run_id: str, ingested_at: datetime):
    offset = 0
    total_count = None
    records = []
    pages_requested = 0
    rows_fetched = 0

    while total_count is None or offset < total_count:
        payload = request_json(
            session,
            endpoint="series/observations",
            params={
                "series_id": series_id,
                "api_key": api_key,
                "file_type": "json",
                "observation_start": start_date.isoformat(),
                "observation_end": end_date.isoformat(),
                "realtime_start": FRED_COMPLETE_REALTIME_START,
                "realtime_end": FRED_COMPLETE_REALTIME_END,
                "units": "lin",
                "output_type": 1,
                "sort_order": "asc",
                "limit": FRED_DEFAULT_LIMIT,
                "offset": offset,
            },
        )

        pages_requested += 1
        total_count = int(payload.get("count", 0))
        limit = int(payload.get("limit", FRED_DEFAULT_LIMIT))
        observations = payload.get("observations", [])
        rows_fetched += len(observations)

        for observation in observations:
            payload_hash = hashlib.sha256(
                json.dumps(observation, sort_keys=True, separators=(",", ":")).encode("utf-8")
            ).hexdigest()
            value_raw = observation.get("value")
            records.append(
                {
                    "series_id": series_id,
                    "observation_date": parse_iso_date("observation_date", observation["date"]),
                    "realtime_start": parse_iso_date("realtime_start", observation["realtime_start"]),
                    "realtime_end": parse_iso_date("realtime_end", observation["realtime_end"]),
                    "value_raw": value_raw,
                    "value": parse_value(value_raw),
                    "ingested_at": ingested_at,
                    "run_id": run_id,
                    "payload_hash": payload_hash,
                }
            )

        if not observations:
            break
        offset += limit

    return records, {"api_rows_fetched": rows_fetched, "request_pages": pages_requested}


def collect_counts(df, key_column: str):
    return {
        row[key_column]: int(row["row_count"])
        for row in df.groupBy(key_column).count().withColumnRenamed("count", "row_count").collect()
    }


dbutils.widgets.text("catalog", "market_macro")
dbutils.widgets.text("series_ids", "CPIAUCSL,FEDFUNDS,GDP")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("lookback_days", "90")
dbutils.widgets.text("run_id", str(uuid.uuid4()))
dbutils.widgets.text("secret_scope", "")
dbutils.widgets.text("secret_key", "fred_api_key")


In [ ]:
catalog = dbutils.widgets.get("catalog").strip() or "market_macro"
series_ids = parse_series_ids(dbutils.widgets.get("series_ids").strip())
mode = dbutils.widgets.get("mode").strip().lower()
run_id = dbutils.widgets.get("run_id").strip()
secret_scope = dbutils.widgets.get("secret_scope").strip()
secret_key = dbutils.widgets.get("secret_key").strip() or "fred_api_key"

if not secret_scope:
    raise ValueError("secret_scope is required so the notebook can read the FRED API key from Databricks secrets")

api_key = dbutils.secrets.get(secret_scope, secret_key)
start_date, end_date = resolve_date_window(
    mode=mode,
    start_date_raw=dbutils.widgets.get("start_date").strip(),
    end_date_raw=dbutils.widgets.get("end_date").strip(),
    lookback_days_raw=dbutils.widgets.get("lookback_days").strip(),
)
ingested_at = datetime.now(timezone.utc)

target_table = f"{catalog}.brz_macro.raw_fred_series"
metadata_table = f"{catalog}.brz_macro.raw_fred_series_metadata"

for table_name in [target_table, metadata_table]:
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table {table_name} does not exist. Run 00_platform_setup_catalog_schema.ipynb first."
        )

target_columns = {field.name for field in spark.table(target_table).schema.fields}
if "value_raw" not in target_columns:
    raise RuntimeError(
        f"Target table {target_table} is using the old schema and is missing value_raw. "
        "Drop market_macro.brz_macro.raw_fred_series, rerun 00_platform_setup_catalog_schema.ipynb, then rerun this notebook."
    )

print(
    f"Fetching FRED series {','.join(series_ids)} from {start_date.isoformat()} to {end_date.isoformat()} in {mode} mode"
)

observation_records = []
metadata_records = []
per_series_stats = {}
session = requests.Session()

try:
    for series_id in series_ids:
        metadata_record = fetch_series_metadata(
            session=session,
            api_key=api_key,
            series_id=series_id,
            run_id=run_id,
            ingested_at=ingested_at,
        )
        metadata_records.append(metadata_record)

        series_records, stats = fetch_series_observations(
            session=session,
            api_key=api_key,
            series_id=series_id,
            start_date=start_date,
            end_date=end_date,
            run_id=run_id,
            ingested_at=ingested_at,
        )
        observation_records.extend(series_records)
        per_series_stats[series_id] = {
            **stats,
            "metadata_fetched": 1,
        }
        print(
            f"{series_id}: fetched={stats['api_rows_fetched']} pages={stats['request_pages']}"
        )
finally:
    session.close()

rows_fetched = len(observation_records)
rows_metadata_ready = len(metadata_records)

metadata_schema = StructType([
    StructField("series_id", StringType(), False),
    StructField("title", StringType(), True),
    StructField("frequency", StringType(), True),
    StructField("frequency_short", StringType(), True),
    StructField("units", StringType(), True),
    StructField("units_short", StringType(), True),
    StructField("seasonal_adjustment", StringType(), True),
    StructField("seasonal_adjustment_short", StringType(), True),
    StructField("observation_start", DateType(), True),
    StructField("observation_end", DateType(), True),
    StructField("last_updated", StringType(), True),
    StructField("notes", StringType(), True),
    StructField("ingested_at", TimestampType(), False),
    StructField("run_id", StringType(), False),
    StructField("payload_hash", StringType(), True),
])

metadata_df = spark.createDataFrame(metadata_records, schema=metadata_schema)
metadata_rows_merged = metadata_df.count()

metadata_existing_key_count = (
    metadata_df.select("series_id")
    .join(
        spark.table(metadata_table).select("series_id"),
        on=["series_id"],
        how="inner",
    )
    .count()
)

DeltaTable.forName(spark, metadata_table).alias("tgt").merge(
    metadata_df.alias("src"),
    "tgt.series_id = src.series_id",
).whenMatchedUpdate(
    set={
        "title": "src.title",
        "frequency": "src.frequency",
        "frequency_short": "src.frequency_short",
        "units": "src.units",
        "units_short": "src.units_short",
        "seasonal_adjustment": "src.seasonal_adjustment",
        "seasonal_adjustment_short": "src.seasonal_adjustment_short",
        "observation_start": "src.observation_start",
        "observation_end": "src.observation_end",
        "last_updated": "src.last_updated",
        "notes": "src.notes",
        "ingested_at": "src.ingested_at",
        "run_id": "src.run_id",
        "payload_hash": "src.payload_hash",
    }
).whenNotMatchedInsertAll().execute()

display(metadata_df.orderBy("series_id"))

if not observation_records:
    result = {
        "status": "success_empty",
        "mode": mode,
        "incremental_strategy": "observation-window re-pull for phase-1 correctness",
        "series_ids": series_ids,
        "start_date": start_date.isoformat(),
        "end_date": end_date.isoformat(),
        "target_table": target_table,
        "metadata_table": metadata_table,
        "rows_fetched": 0,
        "rows_after_filter": 0,
        "rows_after_dedup": 0,
        "rows_to_update": 0,
        "rows_to_insert": 0,
        "rows_merged": 0,
        "rows_metadata_to_update": metadata_existing_key_count,
        "rows_metadata_to_insert": metadata_rows_merged - metadata_existing_key_count,
        "rows_metadata_merged": metadata_rows_merged,
        "run_id": run_id,
        "per_series_stats": per_series_stats,
    }
else:
    observation_schema = StructType([
        StructField("series_id", StringType(), False),
        StructField("observation_date", DateType(), False),
        StructField("realtime_start", DateType(), False),
        StructField("realtime_end", DateType(), False),
        StructField("value_raw", StringType(), True),
        StructField("value", DoubleType(), True),
        StructField("ingested_at", TimestampType(), False),
        StructField("run_id", StringType(), False),
        StructField("payload_hash", StringType(), True),
    ])

    raw_df = spark.createDataFrame(observation_records, schema=observation_schema)
    date_filtered_df = raw_df.filter(
        (F.col("observation_date") >= F.lit(start_date)) & (F.col("observation_date") <= F.lit(end_date))
    )
    rows_after_filter = date_filtered_df.count()

    dedup_window = Window.partitionBy(
        "series_id",
        "observation_date",
        "realtime_start",
        "realtime_end",
    ).orderBy(
        F.col("ingested_at").desc(),
        F.col("payload_hash").desc(),
    )

    deduped_df = (
        date_filtered_df
        .withColumn("_row_number", F.row_number().over(dedup_window))
        .filter(F.col("_row_number") == 1)
        .drop("_row_number")
    )

    rows_after_dedup = deduped_df.count()
    existing_key_count = (
        deduped_df.select("series_id", "observation_date", "realtime_start", "realtime_end")
        .join(
            spark.table(target_table).select("series_id", "observation_date", "realtime_start", "realtime_end"),
            on=["series_id", "observation_date", "realtime_start", "realtime_end"],
            how="inner",
        )
        .count()
    )

    DeltaTable.forName(spark, target_table).alias("tgt").merge(
        deduped_df.alias("src"),
        "tgt.series_id = src.series_id AND "
        "tgt.observation_date = src.observation_date AND "
        "tgt.realtime_start = src.realtime_start AND "
        "tgt.realtime_end = src.realtime_end",
    ).whenMatchedUpdate(
        set={
            "value_raw": "src.value_raw",
            "value": "src.value",
            "ingested_at": "src.ingested_at",
            "run_id": "src.run_id",
            "payload_hash": "src.payload_hash",
        }
    ).whenNotMatchedInsertAll().execute()

    display(deduped_df.orderBy("series_id", "observation_date", "realtime_start", "realtime_end"))

    result = {
        "status": "success",
        "mode": mode,
        "incremental_strategy": "observation-window re-pull for phase-1 correctness",
        "series_ids": series_ids,
        "start_date": start_date.isoformat(),
        "end_date": end_date.isoformat(),
        "target_table": target_table,
        "metadata_table": metadata_table,
        "rows_fetched": rows_fetched,
        "rows_after_filter": rows_after_filter,
        "rows_after_dedup": rows_after_dedup,
        "rows_to_update": existing_key_count,
        "rows_to_insert": rows_after_dedup - existing_key_count,
        "rows_merged": rows_after_dedup,
        "rows_metadata_to_update": metadata_existing_key_count,
        "rows_metadata_to_insert": metadata_rows_merged - metadata_existing_key_count,
        "rows_metadata_merged": metadata_rows_merged,
        "run_id": run_id,
        "per_series_stats": per_series_stats,
    }

result
